# GSE138458 — Classical Baselines

Five regularized/ensemble baselines, evaluated the same way Day 1 established: repeated, subject-grouped, stratified CV — never a single split, never plain accuracy. This notebook picks the best performer and runs SHAP on it, to set up the Day 3-4 question: does the autoencoder's latent space find anything these interpretable models don't?

Reusable training/evaluation/interpretability code lives in [`biomedical_ml.models`](../src/biomedical_ml/models.py), [`biomedical_ml.evaluation`](../src/biomedical_ml/evaluation.py), and [`biomedical_ml.shap_utils`](../src/biomedical_ml/shap_utils.py) — imported and unit-tested. This notebook is results inspection, SHAP plots, and narrative.

In [ ]:
from __future__ import annotations

import json
import warnings

import matplotlib.pyplot as plt
import seaborn as sns

from biomedical_ml.config import FIGURES_DIR, RESULTS_DIR, SEED, ensure_dirs, load_config, set_seed
from biomedical_ml.evaluation import best_model, evaluate_models, summarize_results
from biomedical_ml.models import MODEL_NAMES, build_pipeline
from biomedical_ml.preprocessing import build_dataset
from biomedical_ml.shap_utils import compute_shap_values, top_shap_genes
from biomedical_ml.splits import holdout_split

set_seed()
ensure_dirs()
sns.set_theme(style="whitegrid", font_scale=0.9)
%matplotlib inline

# Benign: RandomForestClassifier's internal n_jobs=-1 joblib parallelism warns
# on every one of the 25 CV folds otherwise, which drowns out real output.
warnings.filterwarnings("ignore", message=".*sklearn.utils.parallel.delayed.*")

cfg = load_config()
K = cfg["features"]["k"]
N_SPLITS = cfg["split"]["n_splits"]
N_REPEATS = cfg["split"]["n_repeats"]
print(f"config: k={K}, n_splits={N_SPLITS}, n_repeats={N_REPEATS} (from configs/default.yaml)")

MODEL_COLOURS = dict(
    zip(
        ["logreg_l2", "logreg_l1", "logreg_elasticnet", "random_forest", "xgboost"],
        sns.color_palette("deep", 5),
        strict=True,
    )
)

## Load the dataset

Same `build_dataset` as Day 1: 330 samples, 307 SLE / 23 control, restricted to annotated probes.

In [ ]:
dataset = build_dataset(annotated_only=True)
print(dataset.summary())

## Five baselines, one pipeline shape

Every model is `SelectKBest(k=2000) -> StandardScaler -> classifier`, refit inside every CV fold so feature selection never sees the test fold. Logistic regression covers the regularization spectrum via `l1_ratio` (the current scikit-learn API — `penalty=` is deprecated): L2, L1, and elastic net. Random Forest and XGBoost add non-linear, ensemble baselines.

F1 and balanced accuracy need a decision threshold, and a blind 0.5 cutoff understates a model at 13:1 imbalance — [`evaluation._best_threshold`](../src/biomedical_ml/evaluation.py) tunes it per fold on the training data's own predictions only, never on the held-out fold. ROC-AUC and PR-AUC are threshold-free and unaffected.

In [ ]:
results = evaluate_models(
    list(MODEL_NAMES), dataset.X, dataset.y, dataset.groups,
    k=K, n_splits=N_SPLITS, n_repeats=N_REPEATS, seed=SEED,
)
results.to_csv(RESULTS_DIR / "baseline_cv_results.csv", index=False)

summary = summarize_results(results)
summary.round(3)

All five baselines land in a tight band: mean ROC-AUC from 0.943 (Random Forest) to 0.970 (L2 logistic regression), with `xgboost` a close second at 0.968. That is not a surprise given Day 1's supervised check (a plain L2 logistic regression already reached ROC-AUC ≈ 0.96) — it confirms the separability is robust across regularization styles and model families, not an artefact of one particular classifier.

Two things are worth flagging rather than glossing over:

- **The ranking is well within fold-to-fold noise.** Every model's ROC-AUC std (0.06-0.09) is larger than the 0.027 gap between the best and worst mean. Day 1 already showed why: 22 control subjects makes single-split estimates noisy, and that noise doesn't vanish just because we're now comparing five models instead of one. "L2 logistic regression is best" should be read as "the linear and boosted-tree baselines are indistinguishable here," not as a confident win.
- **This matches the precedent the project brief anticipated.** Kong & Yu (2018) and similar work on small, high-dimensional (`n << p`) gene expression data found regularized linear models often match or beat more flexible ones. That is exactly what happened: L2 logistic regression — the simplest, cheapest model here — is tied for first.
- **L1 logistic regression has the best balanced accuracy (0.944) despite a lower mean ROC-AUC (0.958).** This is the payoff from per-fold threshold tuning: ROC-AUC (rank-based, threshold-free) and balanced accuracy (threshold-dependent) can disagree, and L1's sparser decision function evidently sits at a better *operating point* even with a slightly noisier ranking.


In [ ]:
order = summary.index.tolist()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, metric, title in zip(
    axes, ["roc_auc", "balanced_accuracy"], ["ROC-AUC", "Balanced accuracy"], strict=True
):
    sns.boxplot(
        data=results, x="model", y=metric, order=order, hue="model", palette=MODEL_COLOURS,
        legend=False, ax=ax,
    )
    sns.stripplot(
        data=results, x="model", y=metric, order=order, color="0.25", size=3, alpha=0.5, ax=ax
    )
    ax.set_title(f"{title} across {N_SPLITS * N_REPEATS} folds")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "baseline_cv_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

This is the Day 1 finding about 22 control subjects showing up again: fold-to-fold spread is visible for every model, not just one. Comparing models on their mean is fine; comparing them on small mean differences is not — with this few control subjects, those differences are well within the fold-to-fold noise shown here.

## SHAP on the best-performing model

SHAP, not permutation importance, because it gives signed, per-feature attributions — the form needed later to compare against whatever axis separates classes in the autoencoder's latent space (Day 4).

SHAP is computed on a single subject-grouped holdout split (fit on train, explain the held-out fold), rather than re-fit inside every CV fold: interpretability here asks "what does *a* trained model rely on", not "how stable is that reliance across resamples" — the CV loop above already answered the stability question for predictive performance.

In [ ]:
chosen_model = best_model(results)
print(f"best model by mean ROC-AUC: {chosen_model}")

train_idx, test_idx = holdout_split(dataset.y, dataset.groups, n_splits=N_SPLITS, seed=SEED)
X_train, X_test = dataset.X.iloc[train_idx], dataset.X.iloc[test_idx]
y_train, y_test = dataset.y.iloc[train_idx], dataset.y.iloc[test_idx]

final_pipeline = build_pipeline(chosen_model, k=K, seed=SEED)
final_pipeline.fit(X_train, y_train)
print(f"held-out ROC-AUC: {final_pipeline.score(X_test, y_test):.3f} (accuracy; see CV numbers above for the metrics that matter here)")

In [ ]:
shap_values, feature_names = compute_shap_values(final_pipeline, X_train, X_test)
top_genes = top_shap_genes(shap_values, feature_names, dataset.annotation, n=20)
top_genes

In [ ]:
plot_data = top_genes.iloc[::-1]  # largest at the top of the barh
labels = plot_data["gene_symbol"].fillna(plot_data["probe_id"])
colours = ["#c44e52" if v >= 0 else "#4c72b0" for v in plot_data["mean_signed_shap"]]

fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(labels, plot_data["mean_abs_shap"], color=colours)
ax.set_xlabel("mean |SHAP value|")
ax.set_title(f"Top 20 features by SHAP importance — {chosen_model}\n(red = pushes toward SLE, blue = pushes toward control)")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "baseline_shap_top_genes.png", dpi=150, bbox_inches="tight")
plt.show()

`logreg_l2` is the chosen model (top mean ROC-AUC, 0.970). The held-out split above shows ROC-AUC = 1.000 — that number is not meaningful on its own (66 samples, only 4 of them control; see Day 1's finding on why single-split estimates are noisy here) and is reported only as a sanity check that this particular fit isn't broken, not as a headline metric.

Two things stand out in the ranking itself:

- **No single gene dominates.** The top bar (CNOT7) sits at mean |SHAP| ≈ 0.16, and the ranking decays gently rather than dropping off a cliff — the top 20 span only 0.16 to 0.07. L2 regularization does what it's designed to do: spread weight across many correlated features rather than concentrate it on a few.
- **The signed values are small relative to the absolute ones.** MCM8, for example, has mean |SHAP| = 0.161 but mean *signed* SHAP of only -0.025 — meaning its push toward SLE vs. control flips direction across samples rather than pointing one way consistently. That is the signature of a feature riding along with a correlated block of genes rather than being an independent, directional biomarker.

Both observations point to the same limitation, worth stating plainly: with 2000 selected features and ~264 training samples on data as collinear as co-regulated gene expression, L2-penalized coefficients (and the SHAP values derived from them) are not a reliable guide to *which individual gene* matters biologically. The model can hit ROC-AUC ≈ 0.97 by spreading small weights over many redundant, co-expressed probes — it does not need to, and does not, pick out "the" driver gene. This is a real constraint on how far classical-model interpretability can be pushed here, not a bug in the SHAP computation, and it is exactly the kind of thing the Day 3-4 latent-space comparison should keep in mind: a non-linear model facing the same collinearity may show the same diffuseness, or it may not — that comparison is itself informative.

## Cross-check against Day 1's unsupervised top-variance genes

Day 1 found that the highest-*variance* genes are dominated by interferon-signature activity (IFI27, IFI44L, RSAD2, IFIT1, OASL, OAS1) — but that axis varies *within* the SLE group and does not separate SLE from control in PCA. SHAP now gives a *supervised* ranking: which genes the model actually leans on to tell SLE from control. If these lists barely overlap, it's further evidence that disease status lives in a different direction than the dataset's dominant unsupervised variance.

In [ ]:
eda_summary_path = RESULTS_DIR / "eda_summary.json"
eda_summary = json.loads(eda_summary_path.read_text(encoding="utf-8"))
top_variance_genes = set(eda_summary["top_variable_genes"])
top_shap_gene_set = set(top_genes["gene_symbol"].dropna())

overlap = top_variance_genes & top_shap_gene_set
print(f"Day 1 top-variance genes (25): {sorted(top_variance_genes)}")
print(f"Day 2 top-SHAP genes ({len(top_shap_gene_set)}): {sorted(top_shap_gene_set)}")
print(f"\nOverlap: {sorted(overlap)} ({len(overlap)} genes)")

**Zero overlap.** None of the 20 top-SHAP genes appear in Day 1's 25 top-variance genes, and vice versa.

That is a clean, if slightly uncomfortable, result. Day 1 already showed that global unsupervised variance (dominated by interferon-signature activity — IFI27, IFI44L, RSAD2, OAS1...) does not separate SLE from control in PCA. This extends that finding down to the gene level: the genes driving the model's *supervised* decision aren't the genes with the most raw variance, and they also aren't the well-known interferon/HLA genes that the SLE literature usually points to.

Given the collinearity caveat above, the honest reading is not "the model discovered a novel disease axis instead of the interferon signature." It is closer to: **L2 logistic regression with 2000 correlated features doesn't need to touch the "obvious" biomarker genes to hit ROC-AUC ≈ 0.97, because there is enough redundant signal spread across the transcriptome that many different gene subsets could do the job.** That is itself informative for Day 3-4 — it means "did the autoencoder find the interferon signature?" and "did the autoencoder find the same genes SHAP flagged?" are two different, both worth-asking questions, and a "no" to either one wouldn't be surprising given what Day 1-2 already show about how distributed this signal is.

## Summary — what this means for Day 3 onward

1. **All five baselines land within 0.943-0.970 mean ROC-AUC — a gap smaller than the fold-to-fold noise.** `logreg_l2` narrowly wins, but "the simplest, cheapest model is tied for first" is the finding, not "L2 beat XGBoost." This matches the precedent the project brief anticipated (Kong & Yu 2018): on `n << p` gene expression data, regularized linear models are competitive with, not inferior to, more flexible ones.
2. **F1/balanced accuracy needed a tuned threshold, and it mattered**: `logreg_l1` has the best balanced accuracy (0.944) despite a slightly lower mean AUC (0.958) than the top model — ranking quality and operating-point quality aren't the same thing at this class ratio.
3. **SHAP on the best model surfaces a diffuse signal with zero overlap against Day 1's top-variance genes** — and, notably, none of the classic interferon/HLA genes that dominate this disease's literature. The likely explanation isn't a novel disease axis; it's that 2000 collinear features give an L2 model many equally-good ways to reach ROC-AUC ≈ 0.97, so *which* genes it leans on is somewhat arbitrary. That collinearity problem doesn't go away for the autoencoder — but the AE isn't penalized the same way, so its latent space might concentrate signal differently. Worth checking directly rather than assuming.
4. **The bar the autoencoder has to clear is now concrete: ROC-AUC ≈ 0.94-0.97**, not "beat a strawman." If the AE's latent space, probed with a simple linear classifier, lands meaningfully below that band, that is itself a legitimate, citable-pattern finding for this kind of small, high-dimensional data — not a failed pipeline (see the project brief's scope-protection rule).

The cell below saves everything a later notebook needs: the CV summary, the chosen model, and its top SHAP genes.

In [ ]:
baseline_summary = {
    "config": {"k": K, "n_splits": N_SPLITS, "n_repeats": N_REPEATS},
    "cv_summary": {
        model: {col: round(float(val), 4) for col, val in row.items()}
        for model, row in summary.to_dict(orient="index").items()
    },
    "best_model": chosen_model,
    "top_shap_genes": top_genes.to_dict(orient="records"),
    "overlap_with_day1_top_variance_genes": sorted(overlap),
}

summary_path = RESULTS_DIR / "baseline_summary.json"
summary_path.write_text(json.dumps(baseline_summary, indent=2), encoding="utf-8")
print(f"wrote {summary_path}")
print(f"wrote {RESULTS_DIR / 'baseline_cv_results.csv'}")